<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/datasetImballenced_problem_fixed_test2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [2]:
import os
import shutil
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

from sklearn.utils.class_weight import compute_class_weight


In [3]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [4]:
BASE = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"

MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/best_model_improved.keras"


In [6]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

shutil.copytree(IMG_SRC, BASE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
batch_size = 16
image_size = 256
FUSION_LAYER = "block4c_add"

In [6]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.02),
    layers.RandomZoom(0.02),
])

In [7]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)

    gray = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)

    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge = edge / (tf.reduce_max(edge) + 1e-6)

    rgb = preprocess_input(image)

    return (rgb, edge), label

In [8]:
def prepare_dataset(path, shuffle):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=shuffle,
        seed=SEED
    )

    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = prepare_dataset(f"{BASE}/train", True)
val_ds = prepare_dataset(f"{BASE}/valid", False)
test_ds = prepare_dataset(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [9]:
loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0)


In [10]:
def create_model():

    rgb_input = layers.Input(shape=(image_size, image_size, 3))
    edge_input = layers.Input(shape=(image_size, image_size, 1))

    # RGB branch
    x_rgb = data_augmentation(rgb_input)
    x_rgb = preprocess_input(x_rgb)

    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(image_size, image_size, 3)
    )

    for layer in base_model.layers[:-160]:
        layer.trainable = False
    for layer in base_model.layers[-160:]:
        layer.trainable = True

    fusion_layer = base_model.get_layer(FUSION_LAYER)

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=fusion_layer.output
    )

    middle_feature = feature_extractor(x_rgb)

    # Edge branch
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(edge_input)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)

    x = layers.Resizing(
        middle_feature.shape[1],
        middle_feature.shape[2]
    )(x)

    x = layers.Conv2D(middle_feature.shape[-1], 1)(x)

    # Fusion
    fused = layers.Concatenate()([middle_feature, x])

    fused = layers.Conv2D(
        256, 3,
        activation="relu",
        padding="same",
        kernel_regularizer=l2(1e-5)
    )(fused)

    # Attention (simple but stable)
    att = layers.GlobalAveragePooling2D()(fused)
    att = layers.Dense(256, activation="sigmoid")(att)

    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Concatenate()([fused, att])

    # Classifier
    fused = layers.Dense(128, activation="relu")(fused)
    fused = layers.Dropout(0.3)(fused)

    outputs = layers.Dense(2, activation="softmax")(fused)

    model = tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=loss_fn,
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ]
    )

    return model

In [11]:
model = create_model()
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        320 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 256, 256,  │          0 │ input_layer[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 16, 16,    │  1,317,880 │ sequential[0][0]  │
│ (Functional)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │     16,512 │ resizing[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ functional_1[0][… │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │    590,080 │ concatenate[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ conv2d_4[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ conv2d_4[0][0]  

 Total params: 2,149,242 (8.20 MB)

 Trainable params: 831,170 (3.17 MB)

 Non-trainable params: 1,318,072 (5.03 MB)

In [12]:
checkpoint = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_improved.keras",
    monitor="val_auc",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_auc",
    patience=10,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.8589 - auc: 0.9222 - loss: 0.0320 - precision: 0.8589 - recall: 0.8589
Epoch 1: val_auc improved from None to 0.95520, saving model to /drive/MyDrive/checkpoints/best_improved.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_improved.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 165s 296ms/step - accuracy: 0.8767 - auc: 0.9372 - loss: 0.0250 - precision: 0.8767 - recall: 0.8767 - val_accuracy: 0.8891 - val_auc: 0.9552 - val_loss: 0.0209 - val_precision: 0.8891 - val_recall: 0.8891
Epoch 2/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step - accuracy: 0.8878 - auc: 0.9525 - loss: 0.0209 - precision: 0.8878 - recall: 0.8878
Epoch 2: val_auc improved from 0.95520 to 0.95629, saving model to /drive/MyDrive/checkpoints/best_improved.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_improved.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 141s 280ms/step - accuracy: 0.8904 - auc: 0.9540 - loss: 0.0206 - pre

In [14]:
results = model.evaluate(test_ds)

print("\n==============================")
print("FINAL RESULTS")
print("==============================")
print("Loss, Accuracy, AUC, Precision, Recall:", results)


63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 250ms/step - accuracy: 0.9032 - auc: 0.9678 - loss: 0.0190 - precision: 0.9032 - recall: 0.9032

FINAL RESULTS
Loss, Accuracy, AUC, Precision, Recall: [0.01902039907872677, 0.9031935930252075, 0.9677994847297668, 0.9031935930252075, 0.9031935930252075]


In [15]:
model.save(MODEL_SAVE_PATH)
print("Model saved!")

Model saved!
